# 항 단위 조각 임베딩 (Colab GPU)

`rag_rechunk.py`로 만든 `data/hang/chunks.jsonl`을 BGE-M3로 임베딩해요. 런타임 → 런타임 유형 변경 → **GPU (T4 이상)**.

1. 아래 셀을 차례로 실행하고, 업로드 창이 뜨면 내 컴퓨터의 `DB/data/hang/chunks.jsonl`을 골라요.
2. 끝나면 `hang_embeddings.zip`이 받아져요. 압축을 풀어서 나온 파일 4개를 `DB/data/hang/`에 넣어요.
3. 내 컴퓨터 DB 폴더에서 `..\backend\.venv\Scripts\python rag_load_hang.py`를 실행해요. 임베딩은 다시 계산하지 않고 DB에 넣기만 해요.

In [ ]:
!pip -q install -U sentence-transformers

In [ ]:
from google.colab import files
up = files.upload()            # DB/data/hang/chunks.jsonl
assert "chunks.jsonl" in up, "chunks.jsonl 파일을 올려 주세요"

In [ ]:
import json, time, numpy as np, torch
from sentence_transformers import SentenceTransformer

chunks = [json.loads(l) for l in open("chunks.jsonl", encoding="utf-8")]
target = [c for c in chunks if c["searchable"]]

def doc_text(c):   # rag_embed.py · rag_load_hang.py 와 같은 입력 형식 (바꾸면 재사용이 안 돼요)
    head = f"{c['doc_name']} {c['article']} {c['article_title']}".strip()
    return f"{head}\n{c['text']}"

texts = [doc_text(c) for c in target]
device = "cuda" if torch.cuda.is_available() else "cpu"
print(len(target), "개,", device)
model = SentenceTransformer("BAAI/bge-m3", device=device)
model.max_seq_length = 4096
if device == "cuda":
    model.half()
t0 = time.time()
emb = model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True, convert_to_numpy=True).astype(np.float32)
print(emb.shape, f"{time.time()-t0:.0f}초")

In [ ]:
import zipfile
np.save("embeddings.npy", emb)
json.dump([c["chunk_id"] for c in target], open("embedding_ids.json", "w", encoding="utf-8"))
json.dump(texts, open("embedding_texts.json", "w", encoding="utf-8"), ensure_ascii=False)
json.dump({"model": "BAAI/bge-m3", "dim": int(emb.shape[1]), "count": len(target), "normalized": True,
           "input": "규정명 조문 제목\\n본문", "device": device + " (colab, fp16)"},
          open("embedding_meta.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
with zipfile.ZipFile("hang_embeddings.zip", "w") as z:
    for f in ["embeddings.npy", "embedding_ids.json", "embedding_texts.json", "embedding_meta.json"]:
        z.write(f)
files.download("hang_embeddings.zip")